# Quantitative Evaluation: Deep Learning Surrogate Modeling for Prop Firm Pass-Rate Frontiers

This notebook implements a deep neural network surrogate model to predict prop firm evaluation passing probabilities. By training a multi-layer perceptron (MLP) framework on synthetic distributions, we bypass computationally expensive Monte Carlo simulations to instantly map non-linear profitability horizons across Risk-Reward (RR) and Win Rate (WR) dimensions.

This notebook does NOT go into the math behind things, if you're interested in that check out the .md that comes with this notebook.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

## 1. Stochastic Account Simulations (Data Engine)

This section contains the monte-carlo simulation loops. Proprietary trading firms impose rigorous mathematical constraints that create severe, non-linear survival boundaries for a portfolio's equity curve. 

### Rules Engine & Algorithmic Mechanics

Rules are set as per FTMO Evaluations. The function returns the number of accounts that passed. 

In [4]:
def simulate_compounding_equity(
    p: float,
    paths: int,
    rr: float,
    risk: float,
    init_cap: float,
    max_loss_pct: float,
    max_daily_loss_pct: float,
    max_daily_trades: int,
    pass_threshold_pct: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, int, int]:
    rng = np.random.default_rng()

    lifetime_threshold = init_cap * (1.0 - max_loss_pct)
    lifetime_pass_target = init_cap * (1.0 + pass_threshold_pct)
    nominal_daily_risk = max_daily_loss_pct * init_cap

    current_equity = np.full(paths, init_cap)
    lifetime_breached = np.zeros(paths, dtype=bool)
    lifetime_passed = np.zeros(paths, dtype=bool)
    daily_breached = np.zeros(paths, dtype=bool)
    trade_counts = np.zeros(paths, dtype=int)

    equity_history = [current_equity.copy()]

    while True:
        num_trades = rng.integers(0, max_daily_trades + 1)
        if num_trades == 0:
            continue

        day_start_equity = current_equity.copy()
        daily_threshold = day_start_equity - nominal_daily_risk
        daily_breached[:] = False

        for _ in range(num_trades):
            raw_outcomes = rng.random(size=paths) < p
            trade_returns = np.where(raw_outcomes, risk * rr, -risk)

            active_mask = ~(lifetime_breached | lifetime_passed)

            current_equity = np.where(
                active_mask,
                current_equity * (1.0 + trade_returns),
                current_equity
            )
            trade_counts += active_mask.astype(int)

            lifetime_breached |= (current_equity <= lifetime_threshold)
            daily_breached |= (current_equity <= daily_threshold)
            lifetime_breached |= daily_breached
            lifetime_passed |= (current_equity >= lifetime_pass_target)

            equity_history.append(current_equity.copy())

            if np.all(lifetime_breached | lifetime_passed):
                equity_curves = np.vstack(equity_history)
                max_trades = int(np.mean(trade_counts))
                equity_curves = equity_curves[:max_trades + 1]
                top_path = np.max(equity_curves, axis=1)
                bottom_path = np.min(equity_curves, axis=1)
                mean_path = np.mean(equity_curves, axis=1)
                time_axis = np.arange(max_trades + 1)
                passed_count = int(np.sum(lifetime_passed))
                failed_count = int(np.sum(lifetime_breached))
                return passed_count

## High-Density Uniform Dataset Generation

Here we generate the 5,000 unique parameter combinations using random uniform sampling

In [ ]:
import time
num_samples = 5000  

accounts = 1000
risk = 0.01
initial_capital = 10000
max_loss = 0.1
max_daily_loss = 0.05
max_daily_trades = 10
pass_threshold = 0.1

rr_wr_passrate = []

print(f"🚀 Launching simulation matrix for {num_samples} unique combinations...")
start_time = time.time()

for i in range(num_samples):
    rr = np.random.uniform(0.1, 3.0) # Lower bound, Upper bound
    wr = np.random.uniform(0.1, 1.0)
    
    pass_count = simulate_compounding_equity(
        wr, accounts, rr, risk, initial_capital, 
        max_loss, max_daily_loss, max_daily_trades, pass_threshold
    ) 
    pass_rate = pass_count / accounts
    
    rr_wr_passrate.append((rr, wr, pass_rate))
    
    if (i + 1) % (num_samples // 10) == 0:
        elapsed = time.time() - start_time
        print(f"Progress: {((i + 1) / num_samples):.0%} completed | Time elapsed: {elapsed:.1f}s")

print(f"🏁 Generation complete! Total dataset size: {len(rr_wr_passrate)} samples.")

🚀 Launching simulation matrix for 5000 unique combinations...
Progress: 10% completed | Time elapsed: 5.2s
Progress: 20% completed | Time elapsed: 9.3s
Progress: 30% completed | Time elapsed: 12.3s
Progress: 40% completed | Time elapsed: 17.9s
Progress: 50% completed | Time elapsed: 22.4s
Progress: 60% completed | Time elapsed: 25.3s
Progress: 70% completed | Time elapsed: 28.3s
Progress: 80% completed | Time elapsed: 31.4s
Progress: 90% completed | Time elapsed: 34.2s
Progress: 100% completed | Time elapsed: 37.3s
🏁 Generation complete! Total dataset size: 5000 samples.


## Feature Engineering and Model Architecture
Here we isolate the data into 80/20 train-test split with stratified random indexing and apply empirical standardization from training distributions to optimize NN weight convergence.

We then define the deep Multi-Layer Perceptron using ReLU activation functions.

In [ ]:
df = pd.DataFrame(rr_wr_passrate, columns=['RR', 'WR', 'Pass_Rate'])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Target Execution Device: {str(device).upper()}")

df_validation = pd.DataFrame(rr_wr_passrate, columns=['RR', 'WR', 'Pass_Rate'])
X = df_validation[['RR', 'WR']].values
y = df_validation['Pass_Rate'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)


class PassRateEmulator(nn.Module):
    def __init__(self):
        super(PassRateEmulator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 128),     
            nn.ReLU(),
            nn.Linear(128, 64),    
            nn.ReLU(),
            nn.Linear(64, 32),     
            nn.ReLU(),
            nn.Linear(32, 1)       
        )
        
    def forward(self, x):
        return self.network(x)

nn_model = PassRateEmulator().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(nn_model.parameters(), lr=0.005)


epochs = 2000
print(f"[INFO] Initializing model optimization over {epochs} epochs...")
print("-" * 50)

for epoch in range(epochs):
    nn_model.train()
    
    predictions = nn_model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch [{epoch+1:04d}/{epochs}] | Mean Squared Error: {loss.item():.6f}")

print("-" * 50)
print("[INFO] Model training phase completed.")


nn_model.eval()
with torch.no_grad(): 
    y_pred_tensor = nn_model(X_test_tensor)
    y_pred = y_pred_tensor.cpu().numpy().flatten()

y_pred = np.clip(y_pred, 0.0, 1.0) 

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "=" * 55)
print("             SURROGATE MODEL EVALUATION REPORT            ")
print("=" * 55)
print(f" Coefficient of Determination (R²): {r2:.4f}")
print(f" Mean Absolute Error (MAE):         {mae:.2%}")
print("=" * 55 + "\n")


fig_val = go.Figure()

line_range = np.linspace(0, 1, 100)
fig_val.add_trace(go.Scatter(
    x=line_range, y=line_range,
    mode='lines',
    name='Ideal Calibration',
    line=dict(color='#ff4444', dash='dash', width=2)
))

fig_val.add_trace(go.Scatter(
    x=y_test, y=y_pred,
    mode='markers',
    name='Empirical Predictions',
    marker=dict(color='#00ffcc', size=6, opacity=0.6),
    hovertemplate="Simulated: %{x:.1%}<br>Predicted: %{y:.1%}<extra></extra>"
))

fig_val.update_layout(
    template='plotly_dark',
    paper_bgcolor='#000000',
    plot_bgcolor='#000000',
    title={'text': "Predictive Validation: Simulated vs. Network Predicted Pass Rates", 'x': 0.5, 'xanchor': 'center'},
    xaxis=dict(title="True Simulated Pass Rate", gridcolor='#222222', tickformat='.0%'),
    yaxis=dict(title="Predicted Pass Rate", gridcolor='#222222', tickformat='.0%'),
    height=1000,
    width=1200,
    showlegend=True
)

fig_val.show()

[INFO] Target Execution Device: CUDA
[INFO] Initializing model optimization over 2000 epochs...
--------------------------------------------------
Epoch [0200/2000] | Mean Squared Error: 0.000699
Epoch [0400/2000] | Mean Squared Error: 0.000283
Epoch [0600/2000] | Mean Squared Error: 0.000183
Epoch [0800/2000] | Mean Squared Error: 0.000160
Epoch [1000/2000] | Mean Squared Error: 0.000122
Epoch [1200/2000] | Mean Squared Error: 0.000109
Epoch [1400/2000] | Mean Squared Error: 0.000102
Epoch [1600/2000] | Mean Squared Error: 0.000134
Epoch [1800/2000] | Mean Squared Error: 0.000507
Epoch [2000/2000] | Mean Squared Error: 0.000088
--------------------------------------------------
[INFO] Model training phase completed.

             SURROGATE MODEL EVALUATION REPORT            
 Coefficient of Determination (R²): 0.9993
 Mean Absolute Error (MAE):         0.56%



## Model Validation and Conclusions
As you can see, the coefficient of determination is 0.9993 and the MAE is 0.56% making this a highly accurate replacement for compute-intensive Monte-Carlo sims.

We now plot the parameter space and export model weights for future use.

Now we yes, WE dont have to clock our CPUs just to generate a plot.

In [9]:
model_path = "prop-firm-pass-rate-predictor.pth"
torch.save(nn_model.state_dict(), model_path)

In [10]:
# 1. Create the massive, high-density evaluation grid
dense_rr = np.linspace(df_validation['RR'].min(), df_validation['RR'].max(), 100)
dense_wr = np.linspace(df_validation['WR'].min(), df_validation['WR'].max(), 100)
RR_mesh, WR_mesh = np.meshgrid(dense_rr, dense_wr)

# Flatten and scale the grid coordinates
grid_inputs = np.c_[RR_mesh.ravel(), WR_mesh.ravel()]
grid_inputs_scaled = scaler.transform(grid_inputs)

# 2. Convert to Tensor and smash it through the GPU
grid_tensor = torch.tensor(grid_inputs_scaled, dtype=torch.float32).to(device)

nn_model.eval()
with torch.no_grad():
    predictions_tensor = nn_model(grid_tensor)
    # Pull results back from VRAM to CPU memory for Plotly
    Z_smooth = predictions_tensor.cpu().numpy().reshape(RR_mesh.shape)

Z_smooth = np.clip(Z_smooth, 0.0, 1.0)

# 3. Build the Ultimate Dark Mode Surface Map
fig_smooth = go.Figure(data=[go.Surface(
    x=dense_rr, y=dense_wr, z=Z_smooth,
    colorscale='RdYlGn',
    hovertemplate="<b>RR:</b> %{x:.2f}<br><b>WR:</b> %{y:.1%}<br><b>Pass Rate:</b> %{z:.1%}<extra></extra>",
    colorbar=dict(title=dict(text='Pass Rate', side='top'), tickformat='.0%', x=0.95, len=0.7)
)])

fig_smooth.update_layout(
    template='plotly_dark', paper_bgcolor='#000000', autosize=True, height=850,
    margin=dict(l=0, r=0, b=0, t=50),
    title={'text': "AI-Generated Continuous Pass Rate Frontier", 'y': 0.95, 'x': 0.5, 'xanchor': 'center'},
    scene=dict(
        bgcolor='#000000',
        xaxis=dict(backgroundcolor='#000000', gridcolor='#222222', showbackground=True, title='Risk-Reward Ratio (RR)'),
        yaxis=dict(backgroundcolor='#000000', gridcolor='#222222', showbackground=True, title='Win Rate (WR)'),
        zaxis=dict(backgroundcolor='#000000', gridcolor='#222222', showbackground=True, title='Pass Rate'),
        aspectmode='cube', camera=dict(eye=dict(x=1.4, y=1.4, z=1.1))
    )
)

# Apply your centering CSS rule
from IPython.display import display, HTML
display(HTML("<style>.plotly-graph-div { margin: 0 auto !important; }</style>"))

fig_smooth.show()